# Phase 4 - Record Linkage and Analysis-Ready Evidence Layer

Project: Open Materials Data Pipeline for Hydrogel Bioink Design

This notebook builds a conservative linkage layer from the completed Phase 2 and Phase 3 checkpoints. It does not train a model, infer hidden physical sample identities, or treat filename similarity as proof of experimental linkage. Its purpose is to connect standardised rheology measurements, measurement-point records, image metadata, acquisition context, quality-control evidence, unresolved review items, and transformation decisions into analysis-ready tables for later exploratory analysis.

The guiding question is:

> Can the curated rheology, image metadata, acquisition context, and QC evidence be linked into an analysis-ready materials-informatics evidence layer without inventing physical sample relationships or hiding uncertainty?

The notebook is designed to be reproducible from a fresh Jupyter session:

`Save -> close notebook -> close Jupyter -> reopen -> restart kernel -> run all -> complete successfully`.

## 1. Environment and Project Setup

**Data-science concept.** A reproducible notebook should create all constants, paths, helper functions and configuration during the same top-to-bottom execution. This prevents hidden Jupyter state from affecting the result.

**Materials-informatics concept.** Record linkage is only meaningful when the source files and checkpoint tables are correctly identified. The notebook therefore avoids machine-specific Windows paths and records every input file that is actually used.

**What this section does.** It imports libraries, resolves a portable project root, defines the Phase 4 output folder, and prepares search locations for the uploaded files or repository files.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import re
import sys
import zipfile

import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

NOTEBOOK_NAME = '04_record_linkage_and_analysis_ready_evidence_layer.ipynb'
PHASE_NAME = '4: record linkage and analysis-ready evidence layer'
RUN_UTC = datetime.now(timezone.utc).isoformat()


def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start] + list(start.parents):
        if any((candidate / marker).exists() for marker in ['data', 'upload', 'README.md', '.git']):
            return candidate
    return start


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw' / 'zenodo_19602891'
METADATA_DIR = DATA_DIR / 'metadata' / 'schema_and_quality_control'
PHASE3_DIR = METADATA_DIR / 'phase3_outputs' / 'standardisation_and_quality_checks'
UPLOAD_DIR = PROJECT_ROOT / 'upload'
PHASE4_OUTPUT_DIR = DATA_DIR / 'processed' / 'record_linkage_and_analysis_ready_evidence_layer'
PHASE4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = []
for path in [PHASE3_DIR, METADATA_DIR, RAW_DIR, UPLOAD_DIR, PROJECT_ROOT]:
    if path.exists() and path not in SEARCH_ROOTS:
        SEARCH_ROOTS.append(path)

print(f'Notebook: {NOTEBOOK_NAME}')
print(f'Run UTC: {RUN_UTC}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Phase 4 output folder: {PHASE4_OUTPUT_DIR}')
print('Input search roots:')
for path in SEARCH_ROOTS:
    print(f'  - {path}')

## 2. Controlled Input Resolution

**Data-science concept.** A file-resolution table is a provenance control. It records which physical file was used for each logical input role, including its file size, modified time and SHA256 hash.

**Materials-data concept.** Some uploaded files have Windows duplicate suffixes such as `(2)` or `(6)`. These suffixes are not scientific metadata. Phase 4 can map them to canonical roles, but it must keep the actual filename in the manifest.

**Permitted assumption.** A duplicate-numbered file can be used as the canonical input when it is the only available candidate or the newest/highest-numbered upload for that role.

**Not permitted.** The notebook must not pretend that candidate filename correspondences are confirmed physical sample links.

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


def duplicate_filename_number(name, canonical):
    canonical_path = Path(canonical)
    pattern = rf'^{re.escape(canonical_path.stem)}(?:\((\d+)\))?{re.escape(canonical_path.suffix)}$'
    match = re.match(pattern, name)
    if not match:
        return None
    return int(match.group(1)) if match.group(1) else 0


def path_priority(path, role):
    text = str(path).lower().replace('\\', '/')
    score = 0
    if '/upload/' in text:
        score += 20
    if role == 'phase3' and ('phase3' in text or 'standardisation' in text):
        score += 50
    if role == 'phase2' and 'schema_and_quality_control' in text:
        score += 50
    if role == 'raw' and ('/raw/' in text or 'zenodo_19602891' in text):
        score += 50
    return score


INPUT_SPECS = [
    ('standardisation_run_manifest.json', 'phase3', 'json'),
    ('measurement_values_standardised.csv', 'phase3', 'csv'),
    ('measurement_points_view.csv', 'phase3', 'csv'),
    ('image_records_standardised.csv', 'phase3', 'csv'),
    ('unresolved_review_items.csv', 'phase3', 'csv'),
    ('qc_findings.csv', 'phase3', 'csv'),
    ('qc_observation_links.csv', 'phase3', 'csv'),
    ('input_verification.csv', 'phase3', 'csv'),
    ('standardisation_decisions.csv', 'phase3', 'csv'),
    ('phase3_issue_decisions.csv', 'phase3', 'csv'),
    ('transformation_rules.csv', 'phase3', 'csv'),
    ('acquisition_context_coverage.csv', 'phase3', 'csv'),
    ('validation_summary.csv', 'phase3', 'csv'),
    ('document_evidence_register.csv', 'phase3', 'csv'),
    ('filename_candidates_unconfirmed.csv', 'phase2', 'csv'),
    ('linkage_summary.csv', 'phase2', 'csv'),
    ('worksheet_register.csv', 'phase2', 'csv'),
    ('entity_schema.csv', 'phase2', 'csv'),
    ('deferred_entities.csv', 'phase2', 'csv'),
    ('ALG-Ph_HA-Ph_rheology_data.zip', 'raw', 'zip'),
    ('image_index.csv', 'raw', 'csv'),
]


def matching_files(canonical):
    matches = []
    seen = set()
    for root in SEARCH_ROOTS:
        for path in root.rglob('*'):
            if not path.is_file() or path.resolve() in seen:
                continue
            number = duplicate_filename_number(path.name, canonical)
            if number is not None:
                matches.append((path, number))
                seen.add(path.resolve())
    return matches


def resolve_input(canonical, role, kind):
    matches = matching_files(canonical)
    if not matches:
        searched = '; '.join(str(root) for root in SEARCH_ROOTS)
        raise FileNotFoundError(f'Missing required input {canonical}. Searched: {searched}')
    exact = [(path, number) for path, number in matches if number == 0 and path.name == canonical]
    pool = exact if exact else matches
    pool = sorted(pool, key=lambda item: (path_priority(item[0], role), item[1], item[0].stat().st_mtime, item[0].stat().st_size), reverse=True)
    selected, duplicate_number = pool[0]
    candidate_list = [str(path.relative_to(PROJECT_ROOT)) if path.is_relative_to(PROJECT_ROOT) else str(path) for path, _ in matches]
    note = 'exact canonical filename selected' if selected.name == canonical else f'duplicate-numbered filename mapped to canonical role; selected suffix {duplicate_number}'
    return {
        'canonical_filename': canonical,
        'actual_filename': selected.name,
        'path': selected,
        'relative_path': str(selected.relative_to(PROJECT_ROOT)) if selected.is_relative_to(PROJECT_ROOT) else str(selected),
        'role': role,
        'kind': kind,
        'duplicate_number': duplicate_number,
        'candidate_count': len(matches),
        'candidates_seen': ' | '.join(candidate_list),
        'file_size_bytes': selected.stat().st_size,
        'modified_time_utc': datetime.fromtimestamp(selected.stat().st_mtime, timezone.utc).isoformat(),
        'sha256': sha256_file(selected),
        'status': 'RESOLVED',
        'resolution_note': note,
    }


resolved_inputs = [resolve_input(*spec) for spec in INPUT_SPECS]
PATHS = {item['canonical_filename']: item['path'] for item in resolved_inputs}
input_resolution = pd.DataFrame([{k: (str(v) if isinstance(v, Path) else v) for k, v in item.items() if k != 'path'} for item in resolved_inputs])
input_resolution.to_csv(PHASE4_OUTPUT_DIR / 'phase4_input_file_resolution.csv', index=False)

print(input_resolution[['canonical_filename', 'actual_filename', 'role', 'candidate_count', 'resolution_note']].to_string(index=False))

## 3. Load Inputs and Verify Phase 3 Checkpoint Invariants

**Data-science concept.** Checkpoint verification prevents downstream tables from being built on the wrong version of the data. Row counts and required columns are treated as critical controls.

**Rheology and hydrogel concept.** The rheology dataset has a hierarchy: source workbook, worksheet, measurement point and measured value. Phase 4 keeps these levels separate so repeated acquisition points are not mistaken for independent samples.

**What this section can establish.** It can confirm that the Phase 3 and Phase 2 outputs are structurally ready for linkage.

**What this section cannot establish.** It cannot prove physical sample identity across rheology and image records.

In [ ]:
def read_csv_input(canonical):
    return pd.read_csv(PATHS[canonical], low_memory=False)


with PATHS['standardisation_run_manifest.json'].open('r', encoding='utf-8') as handle:
    phase3_manifest = json.load(handle)

measurement_values = read_csv_input('measurement_values_standardised.csv')
measurement_points = read_csv_input('measurement_points_view.csv')
image_records = read_csv_input('image_records_standardised.csv')
unresolved_review_items = read_csv_input('unresolved_review_items.csv')
qc_findings = read_csv_input('qc_findings.csv')
qc_observation_links = read_csv_input('qc_observation_links.csv')
input_verification = read_csv_input('input_verification.csv')
standardisation_decisions = read_csv_input('standardisation_decisions.csv')
phase3_issue_decisions = read_csv_input('phase3_issue_decisions.csv')
transformation_rules = read_csv_input('transformation_rules.csv')
acquisition_context_coverage = read_csv_input('acquisition_context_coverage.csv')
validation_summary = read_csv_input('validation_summary.csv')
document_evidence_register = read_csv_input('document_evidence_register.csv')
filename_candidates = read_csv_input('filename_candidates_unconfirmed.csv')
linkage_summary = read_csv_input('linkage_summary.csv')
worksheet_register = read_csv_input('worksheet_register.csv')
entity_schema = read_csv_input('entity_schema.csv')
deferred_entities = read_csv_input('deferred_entities.csv')
raw_image_index = read_csv_input('image_index.csv')

validation_records = []

def add_validation(check, status, evidence, severity='INFO'):
    validation_records.append({'check': check, 'status': status, 'severity': severity, 'evidence': evidence})


def require_columns(frame, name, columns):
    missing = [column for column in columns if column not in frame.columns]
    status = 'PASS' if not missing else 'FAIL'
    add_validation(f'{name} required columns', status, json.dumps({'missing': missing, 'required': columns}), 'CRITICAL')
    if missing:
        raise ValueError(f'{name} is missing required columns: {missing}')


expected_counts = {
    'measurement_values_standardised.csv': 39100,
    'measurement_points_view.csv': 4600,
    'image_records_standardised.csv': 1031,
    'qc_findings.csv': 25,
    'validation_summary.csv': 674,
    'image_index.csv': 1031,
}
observed_counts = {
    'measurement_values_standardised.csv': len(measurement_values),
    'measurement_points_view.csv': len(measurement_points),
    'image_records_standardised.csv': len(image_records),
    'qc_findings.csv': len(qc_findings),
    'validation_summary.csv': len(validation_summary),
    'image_index.csv': len(raw_image_index),
}
for filename, expected in expected_counts.items():
    observed = observed_counts[filename]
    status = 'PASS' if observed == expected else 'FAIL'
    add_validation(f'{filename} checkpoint row count', status, json.dumps({'expected': expected, 'observed': observed}), 'CRITICAL')
    if status == 'FAIL':
        raise ValueError(f'Checkpoint mismatch for {filename}: expected {expected}, observed {observed}')

require_columns(measurement_values, 'measurement_values_standardised', ['member_path', 'sheet', 'source_excel_row', 'source_column_position', 'canonical_field', 'source_value_json', 'standardised_value', 'standardised_unit', 'transformation_status', 'measurement_family', 'qc_status'])
require_columns(measurement_points, 'measurement_points_view', ['member_path', 'sheet', 'source_excel_row', 'measurement_family', 'physical_sample_link_status', 'observation_unit'])
require_columns(image_records, 'image_records_standardised', ['source_csv_record', 'image_id', 'date', 'ink_type_ph', 'shape', 'intensity', 'expected_filename', 'existence_status'])
require_columns(qc_findings, 'qc_findings', ['finding_id', 'rule', 'severity', 'source', 'location', 'evidence', 'action', 'finding_origin'])
require_columns(qc_observation_links, 'qc_observation_links', ['finding_id', 'target_type', 'member_path', 'sheet', 'source_excel_row', 'source_column_position', 'source_csv_record', 'image_field'])
require_columns(filename_candidates, 'filename_candidates_unconfirmed', ['source_stem', 'GG_count', 'viscosity_count', 'GG_members', 'viscosity_members', 'exact_one_each', 'physical_sample_link'])
require_columns(worksheet_register, 'worksheet_register', ['member_path', 'sheet', 'source_stem', 'member_sha256', 'measurement_family', 'header_row', 'point_count', 'physical_sample_link_status'])
require_columns(raw_image_index, 'image_index', ['image_id', 'date', 'ink_type_ph', 'shape', 'intensity', 'completeness_sam', 'contain_sps'])

phase_status = 'PASS' if 'standardisation' in str(phase3_manifest.get('phase', '')).lower() else 'FAIL'
add_validation('Phase 3 manifest phase identity', phase_status, str(phase3_manifest.get('phase')), 'CRITICAL')
if phase_status == 'FAIL':
    raise ValueError('Phase 3 manifest does not look like the standardisation checkpoint.')

print('Checkpoint verification passed.')
for name, frame in [('measurement_values', measurement_values), ('measurement_points', measurement_points), ('image_records', image_records), ('qc_findings', qc_findings), ('validation_summary', validation_summary)]:
    print(f'{name}: {len(frame):,} rows')

## 4. Raw Source Verification and Source-Coordinate Keys

**Data-science concept.** Raw source verification checks that the curated checkpoint can still be traced back to the original Zenodo files. Source-coordinate keys then preserve the row and cell origins of each record.

**Materials concept.** Rheology values such as storage modulus, loss modulus, viscosity, strain, frequency and time are different scientific variables. A long measurement-value table is useful because each source cell can keep its own field, unit, transformation status and QC status.

**Scientific boundary.** These keys identify digital source coordinates. They are not physical sample IDs.

In [ ]:
raw_zip_path = PATHS['ALG-Ph_HA-Ph_rheology_data.zip']
with zipfile.ZipFile(raw_zip_path) as archive:
    zip_members = archive.namelist()
    workbook_members = [name for name in zip_members if name.lower().endswith(('.xlsx', '.xls'))]
add_validation('Raw rheology ZIP readable', 'PASS', json.dumps({'members': len(zip_members), 'workbook_members': len(workbook_members)}), 'CRITICAL')
add_validation('Raw rheology ZIP workbook count matches worksheet register', 'PASS' if len(workbook_members) == len(worksheet_register) else 'WARN', json.dumps({'workbook_members': len(workbook_members), 'worksheet_register_rows': len(worksheet_register)}), 'WARNING')

raw_image_ids = set(raw_image_index['image_id'].dropna().astype(str))
standardised_image_ids = set(image_records['image_id'].dropna().astype(str))
image_id_status = 'PASS' if raw_image_ids == standardised_image_ids else 'FAIL'
add_validation('Raw image_index image IDs match standardised image records', image_id_status, json.dumps({'raw_only': len(raw_image_ids - standardised_image_ids), 'standardised_only': len(standardised_image_ids - raw_image_ids)}), 'CRITICAL')
if image_id_status == 'FAIL':
    raise ValueError('Raw image_index image IDs do not match standardised image records.')


def stable_text(value):
    if pd.isna(value):
        return ''
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value)


def build_key(frame, columns):
    return frame[columns].apply(lambda row: '||'.join(stable_text(value) for value in row), axis=1)


measurement_values = measurement_values.copy()
measurement_points = measurement_points.copy()
worksheet_register = worksheet_register.copy()
filename_candidates = filename_candidates.copy()
image_records = image_records.copy()
raw_image_index = raw_image_index.copy()

measurement_values['measurement_point_key'] = build_key(measurement_values, ['member_path', 'sheet', 'source_excel_row'])
measurement_values['measurement_value_key'] = build_key(measurement_values, ['member_path', 'sheet', 'source_excel_row', 'source_column_position'])
measurement_values['worksheet_key'] = build_key(measurement_values, ['member_path', 'sheet'])
measurement_points['measurement_point_key'] = build_key(measurement_points, ['member_path', 'sheet', 'source_excel_row'])
measurement_points['worksheet_key'] = build_key(measurement_points, ['member_path', 'sheet'])
worksheet_register['worksheet_key'] = build_key(worksheet_register, ['member_path', 'sheet'])
filename_candidates['exact_one_each_bool'] = filename_candidates['exact_one_each'].astype(str).str.lower().isin(['true', '1', 'yes'])

add_validation('Measurement point key uniqueness', 'PASS' if measurement_points['measurement_point_key'].is_unique else 'FAIL', json.dumps({'rows': len(measurement_points), 'unique_keys': measurement_points['measurement_point_key'].nunique()}), 'CRITICAL')
add_validation('Worksheet key uniqueness', 'PASS' if worksheet_register['worksheet_key'].is_unique else 'FAIL', json.dumps({'rows': len(worksheet_register), 'unique_keys': worksheet_register['worksheet_key'].nunique()}), 'CRITICAL')
if not measurement_points['measurement_point_key'].is_unique or not worksheet_register['worksheet_key'].is_unique:
    raise ValueError('Source-coordinate keys are not unique at their required granularity.')

print(f'Raw ZIP workbook members: {len(workbook_members):,}')
print(f'Measurement point keys: {measurement_points["measurement_point_key"].nunique():,}')
print(f'Measurement value keys: {measurement_values["measurement_value_key"].nunique():,}')

## 5. Build Linkage Layers

**Data-science concept.** Linkage should preserve granularity. The measurement-point table has one row per acquisition point, while the measurement-value table has one row per source measurement cell. The image table has one row per image metadata record.

**Polymer, hydrogel and rheology concept.** Candidate filename matches between modulus and viscosity workbooks may help review related rheology files, but they do not prove formulation identity, batch identity, physical sample identity or image-rheology linkage.

**What this section does.** It creates three source-preserving linkage layers and one candidate-only linkage table. It explicitly labels candidate relationships as candidate relationships.

In [ ]:
worksheet_linkage = worksheet_register.merge(
    filename_candidates[['source_stem', 'GG_count', 'viscosity_count', 'exact_one_each_bool', 'physical_sample_link']],
    on='source_stem',
    how='left',
)
worksheet_linkage['rheology_filename_linkage_status'] = worksheet_linkage['exact_one_each_bool'].map({True: 'candidate_by_matching_source_stem', False: 'unresolved_or_not_one_to_one'}).fillna('unresolved_or_not_one_to_one')
worksheet_linkage['confirmed_physical_sample_link'] = False
worksheet_linkage['linkage_caution'] = 'Filename correspondence is candidate evidence only; it is not a confirmed physical sample link.'

qc_links = qc_observation_links.copy()
measurement_cell_links = qc_links[qc_links['target_type'].astype(str).eq('measurement_cell')].copy()
measurement_cell_links['measurement_value_key'] = build_key(measurement_cell_links, ['member_path', 'sheet', 'source_excel_row', 'source_column_position'])
measurement_cell_links['measurement_point_key'] = build_key(measurement_cell_links, ['member_path', 'sheet', 'source_excel_row'])
value_qc_agg = measurement_cell_links.groupby('measurement_value_key')['finding_id'].apply(lambda values: ';'.join(sorted(set(stable_text(value) for value in values if stable_text(value))))).reset_index(name='linked_qc_finding_ids')
point_qc_agg = measurement_cell_links.groupby('measurement_point_key')['finding_id'].apply(lambda values: ';'.join(sorted(set(stable_text(value) for value in values if stable_text(value))))).reset_index(name='point_qc_finding_ids')

worksheet_cols = ['worksheet_key', 'source_stem', 'member_sha256', 'measurement_family', 'header_row', 'point_count', 'physical_sample_link_status', 'GG_count', 'viscosity_count', 'exact_one_each_bool', 'physical_sample_link', 'rheology_filename_linkage_status', 'confirmed_physical_sample_link', 'linkage_caution']

measurement_point_linkage_layer = measurement_points.merge(
    worksheet_linkage[worksheet_cols].rename(columns={'measurement_family': 'worksheet_measurement_family', 'physical_sample_link_status': 'worksheet_physical_sample_link_status'}),
    on='worksheet_key',
    how='left',
).merge(point_qc_agg, on='measurement_point_key', how='left')
measurement_point_linkage_layer['point_qc_finding_ids'] = measurement_point_linkage_layer['point_qc_finding_ids'].fillna('')
measurement_point_linkage_layer['phase4_linkage_status'] = measurement_point_linkage_layer['rheology_filename_linkage_status'].fillna('unresolved')
measurement_point_linkage_layer['phase4_physical_sample_interpretation'] = 'No confirmed physical sample identifier is instantiated in Phase 4.'

linked_measurement_evidence_layer = measurement_values.merge(
    worksheet_linkage[worksheet_cols].rename(columns={'measurement_family': 'worksheet_measurement_family', 'physical_sample_link_status': 'worksheet_physical_sample_link_status'}),
    on='worksheet_key',
    how='left',
).merge(value_qc_agg, on='measurement_value_key', how='left')
linked_measurement_evidence_layer['linked_qc_finding_ids'] = linked_measurement_evidence_layer['linked_qc_finding_ids'].fillna('')


def classify_measurement_value(row):
    qc_text = ' '.join([stable_text(row.get('qc_status')), stable_text(row.get('linked_qc_finding_ids')), stable_text(row.get('qc_finding_ids'))]).lower()
    status = stable_text(row.get('transformation_status')).lower()
    if 'flag' in qc_text and 'no_flag' not in qc_text:
        return 'requires_qc_review_before_analysis'
    if pd.notna(row.get('standardised_value')) and status == 'applied':
        return 'standardised_numeric_safe_for_descriptive_analysis'
    if status == 'text_preserved':
        return 'source_text_preserved_not_numeric'
    return 'unresolved_or_deferred_for_modelling'


linked_measurement_evidence_layer['phase4_analysis_use_status'] = linked_measurement_evidence_layer.apply(classify_measurement_value, axis=1)
linked_measurement_evidence_layer['phase4_linkage_caution'] = 'Rheology filename correspondences remain candidate-level unless independently confirmed.'

candidate_rheology_image_linkage = filename_candidates.copy()
candidate_rheology_image_linkage['rheology_pair_status'] = candidate_rheology_image_linkage['exact_one_each_bool'].map({True: 'candidate_one_modulus_file_and_one_viscosity_file', False: 'candidate_unresolved_file_multiplicity'})
candidate_rheology_image_linkage['confirmed_physical_sample_link'] = False
candidate_rheology_image_linkage['image_linkage_status'] = 'no_direct_image_relationship_supported_by_phase4_evidence'
candidate_rheology_image_linkage['phase4_caution'] = 'Use for manual review and exploratory planning only; do not use as a confirmed sample, formulation, or image link.'

image_left = image_records.copy()
image_right = raw_image_index.copy()
image_left['image_id_merge'] = image_left['image_id'].astype(str)
image_right['image_id_merge'] = image_right['image_id'].astype(str)
image_metadata_linkage_layer = image_left.merge(image_right.add_prefix('raw_'), left_on='image_id_merge', right_on='raw_image_id_merge', how='left')
image_metadata_linkage_layer['metadata_source_match_status'] = image_metadata_linkage_layer['raw_image_id'].notna().map({True: 'matched_to_raw_image_index', False: 'not_matched_to_raw_image_index'})
image_metadata_linkage_layer['image_file_audit_status'] = image_metadata_linkage_layer['existence_status']
image_metadata_linkage_layer['rheology_linkage_status'] = 'no_confirmed_rheology_image_physical_sample_link'
image_metadata_linkage_layer['phase4_use_status'] = 'safe_for_image_metadata_descriptive_analysis; not an image-pixel model input yet'

add_validation('Measurement point linkage preserves row count', 'PASS' if len(measurement_point_linkage_layer) == len(measurement_points) else 'FAIL', json.dumps({'input_rows': len(measurement_points), 'output_rows': len(measurement_point_linkage_layer)}), 'CRITICAL')
add_validation('Linked measurement evidence preserves row count', 'PASS' if len(linked_measurement_evidence_layer) == len(measurement_values) else 'FAIL', json.dumps({'input_rows': len(measurement_values), 'output_rows': len(linked_measurement_evidence_layer)}), 'CRITICAL')
add_validation('Image metadata linkage preserves row count', 'PASS' if len(image_metadata_linkage_layer) == len(image_records) else 'FAIL', json.dumps({'input_rows': len(image_records), 'output_rows': len(image_metadata_linkage_layer)}), 'CRITICAL')
add_validation('Image metadata image_id uniqueness', 'PASS' if image_metadata_linkage_layer['image_id'].is_unique else 'FAIL', json.dumps({'rows': len(image_metadata_linkage_layer), 'unique_image_ids': image_metadata_linkage_layer['image_id'].nunique()}), 'CRITICAL')
add_validation('Candidate table remains candidate-only', 'PASS' if not candidate_rheology_image_linkage['confirmed_physical_sample_link'].any() else 'FAIL', 'No confirmed physical sample links should be generated in Phase 4.', 'CRITICAL')

for table_name, input_rows, output_rows in [
    ('measurement_point_linkage_layer', len(measurement_points), len(measurement_point_linkage_layer)),
    ('linked_measurement_evidence_layer', len(measurement_values), len(linked_measurement_evidence_layer)),
    ('image_metadata_linkage_layer', len(image_records), len(image_metadata_linkage_layer)),
]:
    if input_rows != output_rows:
        raise ValueError(f'{table_name} changed row count from {input_rows} to {output_rows}')

print('Linkage layers created:')
print(f'  linked_measurement_evidence_layer: {len(linked_measurement_evidence_layer):,}')
print(f'  measurement_point_linkage_layer: {len(measurement_point_linkage_layer):,}')
print(f'  image_metadata_linkage_layer: {len(image_metadata_linkage_layer):,}')
print(f'  candidate_rheology_image_linkage: {len(candidate_rheology_image_linkage):,}')

## 6. QC, Unresolved Review Items and Analysis-Readiness

**Data-science concept.** An analysis-ready layer should include explicit review and readiness guidance. This is different from producing a clean-looking table that hides uncertainty.

**Materials concept.** Viscosity-unit uncertainty, deferred formulation identity, unresolved sample identity and unconfirmed image-rheology relationships are scientifically important. They must be carried forward rather than silently resolved.

**What this section does.** It creates QC summaries, unresolved review items, and a field-level analysis-readiness table for Phase 5.

In [ ]:
qc_linkage_summary = qc_findings.merge(qc_observation_links.groupby('finding_id').size().reset_index(name='linked_coordinate_count'), on='finding_id', how='left')
qc_linkage_summary['linked_coordinate_count'] = qc_linkage_summary['linked_coordinate_count'].fillna(0).astype(int)
qc_linkage_summary['phase4_interpretation'] = 'QC finding retained and propagated where source coordinates permit.'

review_rows = []
for idx, row in unresolved_review_items.iterrows():
    review_rows.append({'review_item_id': f'P3-OPEN-{idx + 1:03d}', 'source_table': 'unresolved_review_items.csv', 'topic': row.get('review_topic'), 'status': row.get('status'), 'reason': row.get('phase3_disposition'), 'next_action': row.get('next_action'), 'phase4_disposition': 'carried_forward_as_unresolved'})
for idx, row in deferred_entities.iterrows():
    review_rows.append({'review_item_id': f'P2-DEFERRED-{idx + 1:03d}', 'source_table': 'deferred_entities.csv', 'topic': row.get('entity'), 'status': 'deferred', 'reason': row.get('reason_not_instantiated'), 'next_action': 'Resolve only with direct source evidence or documented manual review.', 'phase4_disposition': 'entity_not_instantiated_in_phase4'})
for idx, item in enumerate([
    ('PHASE4-PHYSICAL-SAMPLE-LINK', 'open', 'No verified physical sample identifier links all rheology and image records.', 'Do not use candidate filename links as confirmed sample labels.'),
    ('PHASE4-IMAGE-FILE-AUDIT', 'deferred', 'Actual image archive is not audited in Phase 4.', 'Optional Phase 6 may inspect images.zip.'),
    ('PHASE4-MODEL-REPRODUCTION', 'deferred', 'Authors trained model artefacts are outside Phase 4 scope.', 'Optional Phase 7 may examine models.zip and csv_data_files_generation.zip.'),
], start=1):
    topic, status, reason, next_action = item
    review_rows.append({'review_item_id': f'P4-REVIEW-{idx:03d}', 'source_table': 'phase4_generated', 'topic': topic, 'status': status, 'reason': reason, 'next_action': next_action, 'phase4_disposition': 'explicitly_retained_for_downstream_review'})
unresolved_linkage_review_items = pd.DataFrame(review_rows)

analysis_readiness_summary = pd.DataFrame([
    {'table': 'linked_measurement_evidence_layer.csv', 'field_or_group': 'standardised_value with standardised_unit', 'readiness_status': 'safe_for_descriptive_analysis_when_phase4_analysis_use_status_allows', 'reason': 'Values retain source coordinates, units and transformation status.', 'phase5_instruction': 'Use only after filtering unresolved or QC-review records as appropriate.'},
    {'table': 'linked_measurement_evidence_layer.csv', 'field_or_group': 'canonical_field', 'readiness_status': 'safe_for_grouping', 'reason': 'Canonical fields come from Phase 3 transformation rules.', 'phase5_instruction': 'Group by canonical field before comparing values.'},
    {'table': 'measurement_point_linkage_layer.csv', 'field_or_group': 'measurement_point_key', 'readiness_status': 'safe_as_source_coordinate_key', 'reason': 'Constructed from member path, worksheet and source row.', 'phase5_instruction': 'Use as an observation key, not as a physical sample identifier.'},
    {'table': 'measurement_point_linkage_layer.csv', 'field_or_group': 'phase4_linkage_status', 'readiness_status': 'candidate_or_unresolved_only', 'reason': 'Filename correspondence is not direct physical sample evidence.', 'phase5_instruction': 'Do not use as a ground-truth label.'},
    {'table': 'image_metadata_linkage_layer.csv', 'field_or_group': 'shape, ink_type_ph, intensity, contain_sps', 'readiness_status': 'safe_for_image_metadata_descriptive_analysis', 'reason': 'Fields are source-preserved from image_index.csv and standardised image records.', 'phase5_instruction': 'Use for metadata summaries; do not infer rheology linkage.'},
    {'table': 'candidate_rheology_image_linkage.csv', 'field_or_group': 'source_stem candidate relationships', 'readiness_status': 'manual_review_only', 'reason': 'Useful for reviewing naming correspondences, but not confirmed sample linkage.', 'phase5_instruction': 'Keep separate from confirmed analysis labels.'},
    {'table': 'qc_linkage_summary.csv', 'field_or_group': 'finding_id and linked coordinates', 'readiness_status': 'required_filtering_context', 'reason': 'QC findings identify records requiring caution.', 'phase5_instruction': 'Report whether records with warnings are included or excluded.'},
    {'table': 'unresolved_linkage_review_items.csv', 'field_or_group': 'all rows', 'readiness_status': 'must_be_carried_forward', 'reason': 'Unresolved issues prevent overclaiming.', 'phase5_instruction': 'Cite these limitations in Phase 5.'},
])

phase4_data_dictionary = pd.DataFrame([
    {'table': table_name, 'field': column, 'source': source, 'interpretation_note': note}
    for table_name, frame, source, note in [
        ('linked_measurement_evidence_layer.csv', linked_measurement_evidence_layer, 'Phase 3 measurement values plus Phase 2 worksheet/candidate evidence', 'Long source-cell-level table; not a physical sample table.'),
        ('measurement_point_linkage_layer.csv', measurement_point_linkage_layer, 'Phase 3 measurement points plus worksheet/candidate evidence', 'One row per measurement point; source-coordinate key only.'),
        ('image_metadata_linkage_layer.csv', image_metadata_linkage_layer, 'Phase 3 image records plus raw image_index.csv', 'Image metadata only; no image-pixel audit in Phase 4.'),
        ('candidate_rheology_image_linkage.csv', candidate_rheology_image_linkage, 'Phase 2 filename candidate table', 'Candidate naming relationships only; not confirmed physical sample linkage.'),
        ('unresolved_linkage_review_items.csv', unresolved_linkage_review_items, 'Phase 2, Phase 3 and Phase 4 review records', 'Issues that remain unresolved or deferred.'),
        ('analysis_readiness_summary.csv', analysis_readiness_summary, 'Phase 4 generated review', 'Guidance for Phase 5 use.'),
    ]
    for column in frame.columns
])

add_validation('Unresolved review items retained', 'PASS' if len(unresolved_linkage_review_items) >= len(unresolved_review_items) else 'FAIL', json.dumps({'phase3_unresolved_rows': len(unresolved_review_items), 'phase4_unresolved_rows': len(unresolved_linkage_review_items)}), 'CRITICAL')
phase4_validation_summary = pd.DataFrame(validation_records)
critical_failures = phase4_validation_summary[(phase4_validation_summary['status'] == 'FAIL') & (phase4_validation_summary['severity'] == 'CRITICAL')]
if not critical_failures.empty:
    print(critical_failures.to_string(index=False))
    raise RuntimeError('Critical Phase 4 validation checks failed. Stop before export.')

print('QC findings retained:', len(qc_linkage_summary))
print('Unresolved review rows:', len(unresolved_linkage_review_items))
print('Critical validation failures:', len(critical_failures))

## 7. Export Phase 4 Outputs

**Data-science concept.** Outputs are written to a dedicated Phase 4 folder so previous Phase 2 and Phase 3 checkpoint files remain unchanged. The run manifest records input hashes and output hashes.

**Materials-informatics concept.** The exported tables prepare the project for Phase 5 exploratory analysis while preserving unresolved sample identity and image-rheology linkage boundaries.

**Phase 6 note.** The README summary also records the planned optional Phase 6 image-file audit, because image metadata curation is included here but image-pixel inspection is deliberately deferred.

In [ ]:
def write_csv_output(frame, filename):
    path = PHASE4_OUTPUT_DIR / filename
    frame.to_csv(path, index=False)
    return path


outputs = {}
outputs['phase4_input_file_resolution.csv'] = PHASE4_OUTPUT_DIR / 'phase4_input_file_resolution.csv'
outputs['linked_measurement_evidence_layer.csv'] = write_csv_output(linked_measurement_evidence_layer, 'linked_measurement_evidence_layer.csv')
outputs['measurement_point_linkage_layer.csv'] = write_csv_output(measurement_point_linkage_layer, 'measurement_point_linkage_layer.csv')
outputs['image_metadata_linkage_layer.csv'] = write_csv_output(image_metadata_linkage_layer, 'image_metadata_linkage_layer.csv')
outputs['candidate_rheology_image_linkage.csv'] = write_csv_output(candidate_rheology_image_linkage, 'candidate_rheology_image_linkage.csv')
outputs['unresolved_linkage_review_items.csv'] = write_csv_output(unresolved_linkage_review_items, 'unresolved_linkage_review_items.csv')
outputs['qc_linkage_summary.csv'] = write_csv_output(qc_linkage_summary, 'qc_linkage_summary.csv')
outputs['analysis_readiness_summary.csv'] = write_csv_output(analysis_readiness_summary, 'analysis_readiness_summary.csv')
outputs['phase4_validation_summary.csv'] = write_csv_output(phase4_validation_summary, 'phase4_validation_summary.csv')
outputs['phase4_data_dictionary.csv'] = write_csv_output(phase4_data_dictionary, 'phase4_data_dictionary.csv')

phase4_readme = f'''
# Phase 4 - Record Linkage and Analysis-Ready Evidence Layer

Run UTC: {RUN_UTC}

## Purpose

Phase 4 builds a conservative linkage layer from completed Phase 2 and Phase 3 outputs. It connects standardised rheology values, measurement-point records, image metadata, worksheet provenance, QC findings, unresolved review items and candidate filename evidence without inventing physical sample relationships.

## Main outputs

- `linked_measurement_evidence_layer.csv`: source-cell-level rheology values with provenance, transformation status, QC context and candidate worksheet linkage.
- `measurement_point_linkage_layer.csv`: one row per measurement point with worksheet context and candidate linkage status.
- `image_metadata_linkage_layer.csv`: image metadata matched back to the raw `image_index.csv` identifiers.
- `candidate_rheology_image_linkage.csv`: candidate naming correspondences only; no confirmed rheology-image physical sample links are asserted.
- `unresolved_linkage_review_items.csv`: open or deferred scientific and linkage issues.
- `analysis_readiness_summary.csv`: guidance on which fields are safe for Phase 5 descriptive analysis, candidate-only exploration or exclusion.
- `phase4_validation_summary.csv`: validation ledger for this run.

## Scientific boundary

Phase 4 does not train predictive models, inspect image pixels, instantiate physical sample IDs, or claim confirmed rheology-image relationships. Filename correspondences are retained as candidate evidence only.

## Optional Phase 6 note

Phase 6 is realistic and useful. This dataset includes `images.zip`, which contains resized 800 x 800 printed-construct images. The image metadata in `image_index.csv` includes image ID, date, ink type/concentration, shape, irradiation intensity, and SPS presence. That means Phase 6 can check whether the image files are present, whether image IDs match the metadata, whether shape labels are balanced, and later whether simple image features relate to printing outcomes.

A good Phase 6 could include:

| Phase 6 section | What it does |
| --- | --- |
| Image archive audit | Count image files, check filename pattern, match `resized_IMG_{{image_id}}.png` to `image_index.csv` |
| Image metadata QC | Check missing labels, duplicate image IDs, shape distribution, and ink formulation distribution |
| Linkage to curated metadata | Connect image records to Phase 4/5 formulation and printing-condition records |
| Basic image inspection | Read dimensions, file integrity, colour mode, and corrupted files |
| Optional simple CV features | Area, brightness, and edge-density style features for exploratory analysis, not deep learning yet |

## Optional Phase 7 note

A later Phase 7 may examine `models.zip`, `csv_data_files_generation.zip`, and the authors' GitHub code for model artefact inventory, fitted-parameter audit, and beta-CVAE image-generation feasibility. This is outside Phase 4 scope.
'''
readme_path = PHASE4_OUTPUT_DIR / 'phase4_readme_summary.md'
readme_path.write_text(phase4_readme, encoding='utf-8')
outputs['phase4_readme_summary.md'] = readme_path

output_rows = {}
for filename, path in outputs.items():
    if filename.endswith('.csv'):
        output_rows[filename] = len(pd.read_csv(path, low_memory=False))
    else:
        output_rows[filename] = None
output_hashes = {filename: sha256_file(path) for filename, path in outputs.items()}

phase4_manifest = {
    'phase': PHASE_NAME,
    'notebook': NOTEBOOK_NAME,
    'run_utc': RUN_UTC,
    'project_root': str(PROJECT_ROOT),
    'python': sys.version,
    'platform': platform.platform(),
    'pandas_version': pd.__version__,
    'input_resolution': input_resolution.to_dict(orient='records'),
    'input_row_counts': observed_counts,
    'phase3_manifest_phase': phase3_manifest.get('phase'),
    'phase3_manifest_notebook': phase3_manifest.get('notebook'),
    'output_folder': str(PHASE4_OUTPUT_DIR),
    'output_files': {filename: str(path) for filename, path in outputs.items()},
    'output_row_counts': output_rows,
    'output_sha256': output_hashes,
    'validation_counts': phase4_validation_summary['status'].value_counts(dropna=False).to_dict(),
    'critical_validation_failures': int(len(critical_failures)),
    'scientific_boundary': 'Candidate filename correspondences are not treated as confirmed physical sample or image-rheology links.',
}
manifest_path = PHASE4_OUTPUT_DIR / 'phase4_run_manifest.json'
manifest_path.write_text(json.dumps(phase4_manifest, indent=2), encoding='utf-8')
outputs['phase4_run_manifest.json'] = manifest_path

summary_rows = [
    {'item': 'Linked measurement evidence rows', 'value': len(linked_measurement_evidence_layer)},
    {'item': 'Measurement point linkage rows', 'value': len(measurement_point_linkage_layer)},
    {'item': 'Image metadata linkage rows', 'value': len(image_metadata_linkage_layer)},
    {'item': 'Candidate rheology linkage rows', 'value': len(candidate_rheology_image_linkage)},
    {'item': 'Unresolved linkage review rows', 'value': len(unresolved_linkage_review_items)},
    {'item': 'QC findings retained', 'value': len(qc_linkage_summary)},
    {'item': 'Critical validation failures', 'value': len(critical_failures)},
]
final_summary = pd.DataFrame(summary_rows)
print(final_summary.to_string(index=False))
print('\nPhase 4 outputs written:')
for filename, path in outputs.items():
    print(f'  - {filename}: {path}')
print(f'  - phase4_run_manifest.json: {manifest_path}')